# MedCompress — CheXpert Multi-Label Chest X-ray
**Author: Abhishek Shekhar**

EfficientNetB0 teacher + student on 5 CheXpert pathology labels.

> All outputs persist on Drive. Completed experiments are skipped automatically.

In [ ]:
import os, sys, time, random, json, warnings
warnings.filterwarnings("ignore")
import numpy as np

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── GPU check ───────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
tf.config.set_visible_devices(gpus[0], 'GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TensorFlow {tf.__version__}  GPU: {gpus[0].name}")

# ── Directories (all outputs persist on Drive) ──────────────────────────────
EXP_NAME    = "chexpert_classification"   # ← overridden per notebook
DRIVE_BASE  = f"/content/drive/MyDrive/medcompress/chexpert_classification"
DATA_DIR    = f"/content/data/chexpert_classification"
RESULTS_DIR = f"{DRIVE_BASE}/results"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
MODELS_DIR  = f"{DRIVE_BASE}/models"
TFLITE_DIR  = f"{DRIVE_BASE}/tflite"
for d in [DRIVE_BASE, DATA_DIR, RESULTS_DIR, CKPT_DIR, MODELS_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

SESSION_START = time.time()
print(f"Drive base : {DRIVE_BASE}")
print("Setup complete ✓")

In [ ]:
%%capture install_out
!pip install -q tensorflow-model-optimization tf2onnx onnx scikit-learn pillow tqdm
import tensorflow_model_optimization as tfmot
import tf2onnx
print("All packages installed ✓")

In [ ]:
# ── Kaggle credentials (run once per session) ──────────────────────────────
import os
KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_JSON):
    from google.colab import files
    print("Upload your kaggle.json API token (from kaggle.com → Settings → API):")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials saved ✓")
else:
    print("Kaggle credentials already present ✓")

In [ ]:
# ── Download CheXpert ───────────────────────────────────────────────────────
CHEX_DIR = f"{DATA_DIR}/chexpert"
os.makedirs(CHEX_DIR, exist_ok=True)
if not os.path.exists(f"{CHEX_DIR}/train.csv"):
    print("Downloading CheXpert-v1.0-small (~11 GB)...")
    !kaggle datasets download -d stanfordmlgroup/chexpert -p {CHEX_DIR}
    !cd {CHEX_DIR} && unzip -q "*.zip" && rm -f *.zip
    print("Download complete ✓")
else:
    print(f"CheXpert cached ✓")

import pandas as pd
# Find train.csv
train_csv = None
for root,dirs,files in os.walk(CHEX_DIR):
    if "train.csv" in files:
        _c=pd.read_csv(os.path.join(root,"train.csv"),nrows=2)
        if "Path" in _c.columns: train_csv=os.path.join(root,"train.csv"); break
df = pd.read_csv(train_csv)
print(f"Dataset: {len(df)} rows")

In [ ]:
# ── Configuration ───────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
SEEDS      = [42, 123, 456, 789, 1024]
LABELS     = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]

CFG = {
    "baseline":       {"epochs":20,"lr":1e-4,"patience":7},
    "qat":            {"epochs":10,"lr":1e-5},
    "student_scratch":{"epochs":25,"lr":1e-4,"patience":10},
    "kd":             {"epochs":25,"lr":1e-4,"patience":10,"temperature":4.0,"alpha":0.7},
}

# U-Ones: -1 → 1, NaN → 0
for lbl in LABELS: df[lbl] = df[lbl].fillna(0).replace(-1,1).astype(np.float32)
if "Frontal/Lateral" in df.columns: df = df[df["Frontal/Lateral"]=="Frontal"].reset_index(drop=True)

# Fix image paths
img_root = os.path.dirname(train_csv)
def fix_path(p):
    p = str(p)
    for pref in ["CheXpert-v1.0-small/","CheXpert-v1.0/"]:
        if pref in p: return os.path.join(img_root, p.split(pref,1)[1])
    return os.path.join(img_root, p)
df["abs_path"] = df["Path"].apply(fix_path)
df = df[df["abs_path"].apply(os.path.exists)].reset_index(drop=True)
print(f"Valid rows: {len(df)}")

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def compute_cls_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob.ravel() >= thr).astype(int)
    y_true = y_true.astype(int).ravel()
    auc  = roc_auc_score(y_true, y_prob.ravel())
    f1   = f1_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return {"auc":float(auc),"f1":float(f1),"sensitivity":float(sens),"specificity":float(spec)}

def export_tflite(model, path, quant="fp32", calib_ds=None, n_calib=200):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant == "int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib_ds:
            samples = []
            for imgs, _ in calib_ds:
                for i in range(len(imgs)):
                    samples.append(imgs[i].numpy())
                    if len(samples) >= n_calib: break
                if len(samples) >= n_calib: break
            def rep():
                for s in samples: yield [np.expand_dims(s,0)]
            conv.representative_dataset = rep
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_input_type  = tf.uint8
            conv.inference_output_type = tf.uint8
    elif quant == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    data = conv.convert()
    with open(path,"wb") as f: f.write(data)
    mb = os.path.getsize(path)/1e6
    print(f"  TFLite {quant}: {os.path.basename(path)} ({mb:.1f} MB)")
    return mb

def eval_tflite(path, dataset, warmup=5):
    interp = tf.lite.Interpreter(model_path=path, num_threads=4)
    interp.allocate_tensors()
    inp_d = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]
    preds, labels, lats = [], [], []
    for imgs, lbls in dataset:
        for i in range(len(imgs)):
            x = np.expand_dims(imgs[i].numpy(), 0)
            if inp_d["dtype"] == np.uint8:
                sc,zp = inp_d["quantization"]; x=(x/sc+zp).astype(np.uint8)
            interp.set_tensor(inp_d["index"], x)
            t0=time.perf_counter(); interp.invoke(); t1=time.perf_counter()
            lats.append((t1-t0)*1000)
            r=interp.get_tensor(out_d["index"])
            if out_d["dtype"]==np.uint8:
                sc,zp=out_d["quantization"]; r=(r.astype(np.float32)-zp)*sc
            preds.append(r.squeeze()); labels.append(lbls[i].numpy())
    lats = np.array(lats[warmup:])
    m = compute_cls_metrics(np.array(labels), np.array(preds))
    m["latency_ms"] = float(np.median(lats))
    m["latency_p95_ms"] = float(np.percentile(lats,95))
    m["size_mb"] = os.path.getsize(path)/1e6
    print(f"  TFLite AUC={m['auc']:.4f}  lat={m['latency_ms']:.1f}ms  sz={m['size_mb']:.1f}MB")
    return m

def done_flag(name): return f"{RESULTS_DIR}/{name}.done"
def is_done(name): return os.path.exists(done_flag(name))
def mark_done(name, result):
    with open(f"{RESULTS_DIR}/{name}.json","w") as f: json.dump(result, f)
    open(done_flag(name),"w").close()
def load_result(name):
    with open(f"{RESULTS_DIR}/{name}.json") as f: return json.load(f)

def train_model(model, train_ds, val_ds, cfg, name, monitor="val_auc", mode="max"):
    """Train with Drive checkpoint + resume. Returns history."""
    ckpt = f"{CKPT_DIR}/{name}_best.keras"
    epoch_f = f"{CKPT_DIR}/{name}_epoch.json"
    initial_epoch = 0
    if os.path.exists(ckpt):
        try:
            model.load_weights(ckpt)
            if os.path.exists(epoch_f):
                initial_epoch = json.load(open(epoch_f)).get("epoch",0)
            print(f"  Resumed from epoch {initial_epoch} ✓")
        except Exception as e:
            print(f"  Checkpoint load failed ({e}), starting fresh")
    class _EpochLog(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            json.dump({"epoch":epoch+1}, open(epoch_f,"w"))
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, save_best_only=True, monitor=monitor, mode=mode, verbose=0),
        tf.keras.callbacks.EarlyStopping(
            patience=cfg.get("patience",7), monitor=monitor, mode=mode,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3, mode=mode, verbose=0),
        _EpochLog()
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=cfg.get("epochs",20), initial_epoch=initial_epoch,
        class_weight=cfg.get("class_weight"), callbacks=cbs, verbose=1)
    return model, history

print("Utilities loaded ✓")

In [ ]:
# ── Data pipeline ───────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.10, random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.10/0.90, random_state=42)
print(f"Splits: train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

def make_ds(dataframe, augment=False, shuffle=False):
    paths  = dataframe["abs_path"].values
    labels = dataframe[LABELS].values.astype(np.float32)
    def load(path, label):
        img = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.image.resize(img, [IMG_SIZE,IMG_SIZE])
        img = tf.cast(img, tf.float32)/127.5 - 1.0
        return img, label
    def aug(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.15)
        img = tf.image.random_contrast(img, 0.85, 1.15)
        return img, label
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle: ds = ds.shuffle(len(paths), reshuffle_each_iteration=True)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment: ds = ds.map(aug, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_df, True, True)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

In [ ]:
# ── Models ──────────────────────────────────────────────────────────────────
from tensorflow import keras; from tensorflow.keras import layers

def build_teacher():
    bb = keras.applications.EfficientNetB0(include_top=False,weights="imagenet",
                                            input_shape=(IMG_SIZE,IMG_SIZE,3))
    for l in bb.layers[:-20]: l.trainable=False
    inp = keras.Input((IMG_SIZE,IMG_SIZE,3))
    x = bb(inp,training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(256,activation="relu")(x); x = layers.Dropout(0.15)(x)
    return keras.Model(inp, layers.Dense(5,activation="sigmoid")(x), name="effb0_chex")

def build_student():
    bb = keras.applications.MobileNetV3Small(include_top=False,weights="imagenet",
                                              input_shape=(IMG_SIZE,IMG_SIZE,3))
    for l in bb.layers[:-10]: l.trainable=False
    inp = keras.Input((IMG_SIZE,IMG_SIZE,3))
    x = bb(inp,training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x); x = layers.Dropout(0.3)(x)
    x = layers.Dense(128,activation="relu")(x); x = layers.Dropout(0.15)(x)
    return keras.Model(inp, layers.Dense(5,activation="sigmoid")(x), name="mv3s_chex")

def eval_multilabel(model, ds, tag=""):
    from sklearn.metrics import roc_auc_score
    preds,lbls=[],[]
    for imgs,labs in ds: preds.append(model(imgs,training=False).numpy()); lbls.append(labs.numpy())
    p=np.concatenate(preds); l=np.concatenate(lbls)
    per_auc = [roc_auc_score(l[:,i],p[:,i]) for i in range(5) if l[:,i].sum()>0]
    mean_auc = np.mean(per_auc)
    print(f"  [{tag}] Mean AUC={mean_auc:.4f}  per-label={[f'{a:.3f}' for a in per_auc]}")
    return {"mean_auc":float(mean_auc), **{f"auc_{LABELS[i]}":float(per_auc[i]) for i in range(len(per_auc))}}

print("Models defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPERIMENTS 1-4 (same pipeline as ISIC but multi-label)
# ══════════════════════════════════════════════════════════════════════════════
import tensorflow_model_optimization as tfmot
baseline_results, scratch_results, kd_results, qat_results = [], [], [], []

for seed in SEEDS:
    set_seed(seed)
    # ─ Baseline ─
    key = f"baseline_s{seed}"
    if is_done(key): baseline_results.append(load_result(key)); print(f"⏩ {key}")
    else:
        m = build_teacher()
        m.compile(optimizer=keras.optimizers.Adam(CFG["baseline"]["lr"]),
                  loss="binary_crossentropy", metrics=["accuracy"])
        m, _ = train_model(m, train_ds, val_ds, CFG["baseline"], key, monitor="val_loss", mode="min")
        metrics = eval_multilabel(m, test_ds, key)
        m.save(f"{MODELS_DIR}/effb0_s{seed}.keras")
        mark_done(key, metrics); baseline_results.append(metrics)

    # ─ Student Scratch ─
    key = f"scratch_s{seed}"
    if is_done(key): scratch_results.append(load_result(key)); print(f"⏩ {key}")
    else:
        s = build_student()
        s.compile(optimizer=keras.optimizers.Adam(CFG["student_scratch"]["lr"]),
                  loss="binary_crossentropy", metrics=["accuracy"])
        s, _ = train_model(s, train_ds, val_ds, CFG["student_scratch"], key, monitor="val_loss", mode="min")
        metrics = eval_multilabel(s, test_ds, key)
        s.save(f"{MODELS_DIR}/mv3s_scratch_s{seed}.keras"); mark_done(key, metrics); scratch_results.append(metrics)

print("\nBaseline mean AUC:", round(np.mean([r["mean_auc"] for r in baseline_results]),4))
print("Scratch mean AUC: ", round(np.mean([r["mean_auc"] for r in scratch_results]),4))

In [ ]:
# ─ KD ───────────────────────────────────────────────────────────────────────
T=CFG["kd"]["temperature"]; ALPHA=CFG["kd"]["alpha"]
teacher = keras.models.load_model(f"{MODELS_DIR}/effb0_s42.keras")

for seed in SEEDS:
    key = f"kd_s{seed}"
    if is_done(key): kd_results.append(load_result(key)); print(f"⏩ {key}"); continue
    print(f"\n── KD seed={seed} ──")
    set_seed(seed)
    student = build_student()
    opt = keras.optimizers.Adam(CFG["kd"]["lr"])
    best_loss, patience_c = 1e9, 0
    ckpt = f"{CKPT_DIR}/kd_s{seed}.keras"
    if os.path.exists(ckpt): student.load_weights(ckpt); print("  Resumed ✓")
    for epoch in range(CFG["kd"]["epochs"]):
        ep_loss,n = 0,0
        for imgs,lbls in train_ds:
            with tf.GradientTape() as tape:
                t_out = teacher(imgs,training=False)
                s_out = student(imgs,training=True)
                def soft(x): return tf.nn.sigmoid(tf.math.log(x/(1-x+1e-7)+1e-7)/T)
                ts,ss = soft(t_out), soft(s_out)
                eps=1e-7
                kl = ts*tf.math.log((ts+eps)/(ss+eps))+(1-ts)*tf.math.log((1-ts+eps)/(1-ss+eps))
                loss = ALPHA*tf.reduce_mean(kl)*(T**2) + (1-ALPHA)*tf.reduce_mean(
                    keras.losses.binary_crossentropy(lbls,s_out))
            grads=tape.gradient(loss,student.trainable_variables)
            opt.apply_gradients(zip(grads,student.trainable_variables)); ep_loss+=loss.numpy();n+=1
        vl = np.mean([keras.losses.binary_crossentropy(lb,student(im,training=False)).numpy()
                      for im,lb in val_ds])
        print(f"  Epoch {epoch+1}: loss={ep_loss/n:.4f} val_loss={vl:.4f}")
        if vl<best_loss: best_loss=vl; patience_c=0; student.save_weights(ckpt)
        else:
            patience_c+=1
            if patience_c>=CFG["kd"]["patience"]: break
    student.load_weights(ckpt)
    metrics=eval_multilabel(student, test_ds, key)
    student.save(f"{MODELS_DIR}/mv3s_kd_s{seed}.keras"); mark_done(key,metrics); kd_results.append(metrics)

kd_auc=np.mean([r["mean_auc"] for r in kd_results])
sc_auc=np.mean([r["mean_auc"] for r in scratch_results])
print(f"\n★ KD gain: {(kd_auc-sc_auc)*100:+.2f}% mean AUC")

In [ ]:
# ─ QAT on KD student ────────────────────────────────────────────────────────
for seed in SEEDS:
    key = f"qat_kd_s{seed}"
    if is_done(key): qat_results.append(load_result(key)); print(f"⏩ {key}"); continue
    base = keras.models.load_model(f"{MODELS_DIR}/mv3s_kd_s{seed}.keras")
    qat_m = tfmot.quantization.keras.quantize_model(base)
    qat_m.compile(optimizer=keras.optimizers.Adam(1e-5),
                  loss="binary_crossentropy",metrics=["accuracy"])
    qat_m.fit(train_ds, validation_data=val_ds, epochs=10, verbose=1)
    stripped=tfmot.quantization.keras.strip_pruning(qat_m)
    p=f"{TFLITE_DIR}/mv3s_kd_qat_int8_s{seed}.tflite"
    conv=tf.lite.TFLiteConverter.from_keras_model(stripped)
    conv.optimizations=[tf.lite.Optimize.DEFAULT]
    with open(p,"wb") as f: f.write(conv.convert())
    mb=os.path.getsize(p)/1e6
    metrics=eval_multilabel(stripped,test_ds,key); metrics["size_mb"]=mb
    mark_done(key,metrics); qat_results.append(metrics)

# Summary table
table = pd.DataFrame([
    {"method":"Baseline","mean_auc":np.mean([r["mean_auc"] for r in baseline_results])},
    {"method":"Student Scratch","mean_auc":np.mean([r["mean_auc"] for r in scratch_results])},
    {"method":"KD","mean_auc":np.mean([r["mean_auc"] for r in kd_results])},
    {"method":"KD+QAT","mean_auc":np.mean([r.get("mean_auc",0) for r in qat_results])},
])
table.to_csv(f"{RESULTS_DIR}/chexpert_results.csv",index=False)
print(table.to_string(index=False))
print(f"\nResults saved to Drive: {RESULTS_DIR}")